# Video Clone — Colab full worker (parity local)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daokimluc/video-clone-colab/blob/main/video_clone_colab_backend.ipynb)

Worker trên Colab hỗ trợ **cùng pipeline inference** với desktop:

| Feature | Colab |
|---------|--------|
| **ASR** (Whisper) | ✅ GPU nếu Runtime GPU |
| **OCR** (EasyOCR) | ✅ |
| **TTS** (edge-tts) | ✅ |
| **Translate** (Google free) | ✅ |
| **Export** mix FFmpeg | desktop local (cần file video gốc) |

**Runtime → Run all** → copy `REMOTE_URL` + `SHARED_SECRET` vào app **Cấu hình → Remote**.

## 1) Install deps + cloudflared + fetch worker

In [ ]:
# GPU-friendly stack
!pip -q install fastapi uvicorn httpx faster-whisper edge-tts deep-translator easyocr

import shutil, os

def ensure_cloudflared():
    if shutil.which('cloudflared'):
        print('cloudflared:', shutil.which('cloudflared')); return
    !wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /tmp/cloudflared
    !cp /tmp/cloudflared /usr/local/bin/cloudflared
    print('cloudflared installed')

ensure_cloudflared()

# Pull latest worker from GitHub (full ASR/OCR/TTS)
!wget -q -O /content/colab_worker.py https://raw.githubusercontent.com/daokimluc/video-clone-colab/main/colab_worker.py
print('worker bytes', os.path.getsize('/content/colab_worker.py'))
!cloudflared --version
import torch
print('CUDA available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2) Shared secret

In [ ]:
import secrets, os
SHARED_SECRET = secrets.token_urlsafe(16)
os.environ['VC_COLAB_SECRET'] = SHARED_SECRET
os.environ['SHARED_SECRET'] = SHARED_SECRET
print('='*60)
print('SHARED_SECRET (app → Shared secret):')
print(SHARED_SECRET)
print('='*60)

## 3) Start full worker (free port 8765 if busy)

In [ ]:
import os, time, socket, subprocess, httpx, importlib.util

WORKER_PORT = 8765
os.environ['VC_WORKER_PORT'] = str(WORKER_PORT)
os.environ['VC_COLAB_SECRET'] = SHARED_SECRET

def free_port(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(('127.0.0.1', port)) != 0:
            print('Port free', port); return
    print('Freeing port', port)
    subprocess.run(['bash','-lc', f'fuser -k {port}/tcp || true'], check=False)
    time.sleep(1)

free_port(WORKER_PORT)

# load colab_worker module
spec = importlib.util.spec_from_file_location('colab_worker', '/content/colab_worker.py')
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
mod.SHARED_SECRET = SHARED_SECRET
mod.WORKER_PORT = WORKER_PORT

import uvicorn, threading

def run():
    uvicorn.run(mod.app, host='127.0.0.1', port=WORKER_PORT, log_level='warning')

threading.Thread(target=run, daemon=True).start()
time.sleep(2)

for i in range(15):
    try:
        r = httpx.get(f'http://127.0.0.1:{WORKER_PORT}/health', headers={'X-VC-Secret': SHARED_SECRET}, timeout=3)
        print('Local health:', r.status_code, r.json())
        c = httpx.get(f'http://127.0.0.1:{WORKER_PORT}/capabilities', headers={'X-VC-Secret': SHARED_SECRET}, timeout=3)
        print('Capabilities:', c.json())
        break
    except Exception as e:
        time.sleep(0.5)
        last=e
else:
    print('Worker failed', last)

## 4) Cloudflare tunnel → REMOTE_URL

In [ ]:
import re, subprocess, time, httpx, shutil

if globals().get('_VC_TUNNEL_PROC'):
    try: _VC_TUNNEL_PROC.terminate()
    except: pass
subprocess.run(['bash','-lc','pkill -f "cloudflared tunnel" || true'], check=False)
time.sleep(0.5)

cmd=['cloudflared','tunnel','--url',f'http://127.0.0.1:{WORKER_PORT}','--no-autoupdate']
print('Starting:', ' '.join(cmd), flush=True)
_VC_TUNNEL_PROC=subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

REMOTE_URL=None
pat=re.compile(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com')
deadline=time.time()+90
while time.time()<deadline and not REMOTE_URL:
    line=_VC_TUNNEL_PROC.stdout.readline()
    if not line:
        if _VC_TUNNEL_PROC.poll() is not None: break
        continue
    line=line.rstrip(); print(line, flush=True)
    m=pat.search(line)
    if m: REMOTE_URL=m.group(0).rstrip('/')

if not REMOTE_URL:
    raise RuntimeError('No trycloudflare URL — restart runtime & Run all')

print('='*60, flush=True)
print('REMOTE_URL =', REMOTE_URL, flush=True)
print('SHARED_SECRET =', SHARED_SECRET, flush=True)
print('App: Backend mode = Remote', flush=True)
print('Features: ASR+OCR+TTS+Translate on Colab; Export local', flush=True)
print('='*60, flush=True)
time.sleep(2)
try:
    r=httpx.get(REMOTE_URL+'/health', headers={'X-VC-Secret':SHARED_SECRET}, timeout=25, follow_redirects=True)
    print('Public health:', r.status_code, r.text[:200], flush=True)
    c=httpx.get(REMOTE_URL+'/capabilities', headers={'X-VC-Secret':SHARED_SECRET}, timeout=25, follow_redirects=True)
    print('Public caps:', c.text[:400], flush=True)
except Exception as e:
    print('Public probe later:', e, flush=True)
print('Keep this runtime running.')

### Desktop
1. Cấu hình → Remote → dán REMOTE_URL + SHARED_SECRET  
2. Clone Video → **Nhận dạng** = ASR trên **Colab GPU**  
3. OCR / TTS cũng remote; **Xuất MP4** vẫn local (ffmpeg + file video)

Cần **Runtime → GPU** (T4) để ASR nhanh.